# 제목과 주제로 나누는 청킹 실험

이번 단계에서는 PDF 52~53쪽만 대상으로 여섯 가지 주제를 나눕니다. 청크는 검색하기 쉽게 묶은 글 조각입니다. 글자 수만으로 자르면 서로 다른 주제가 섞일 수 있으므로, PDF에서 확인한 제목을 경계로 사용합니다.

원문 문장은 요약하지 않습니다. 다음 페이지로 이어진 설명도 같은 주제로 묶고, 확인한 그림은 해당 주제에 연결합니다. 결과는 메모리에서 만들며 청크 파일이나 Supabase에 저장하지 않습니다.

## 수업 기반과 프로젝트 추가 기록 (2026-10-01)

코드의 추가 설명은 주석으로 남겼습니다. 이 표시는 프로젝트 적용 기록이며, 수업에서 개념 자체를 다루지 않았다는 뜻은 아닙니다.

- PDF 52~53쪽의 확인한 여섯 제목으로 주제 경계를 정하고 이어지는 문장을 묶습니다.
- 추가: 꼬리말 제거, 원문 글자 구간 기록, 해당 설명에만 그림 연결. 원문을 요약해 대체하지 않습니다.

전체 목록: [수업 기반·추가 적용 기록](../src/car_search_rag/zzong_santafe_lag/learning_additions.md)


### 1. 실험할 두 페이지 읽기

이 셀은 PDF 52~53쪽의 글을 읽습니다. `page_texts`에 담긴 원문은 이후 정리 과정에서도 그대로 둡니다. PDF 쪽수와 매뉴얼에 인쇄된 쪽수는 서로 다르므로 따로 기록합니다.

In [1]:
# [프로젝트 추가 기록] PDF 52~53쪽의 확인한 여섯 제목으로 주제 경계를 정하고 이어지는 문장을 묶습니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
from pathlib import Path
import re
from pypdf import PdfReader

# 실행 위치가 notebooks 폴더여도 프로젝트의 data 폴더를 찾습니다.
project_folder = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
pdf_path = project_folder / "data" / "santafe_hev_manual.pdf"
reader = PdfReader(str(pdf_path))

# PDF 파일은 52~53쪽, 매뉴얼에 인쇄된 쪽수는 47~48쪽입니다.
manual_page_numbers = {52: 47, 53: 48}
page_texts = {number: reader.pages[number - 1].extract_text() or "" for number in [52, 53]}

for number, text in page_texts.items():
    print(f"PDF {number}쪽 / 매뉴얼 {manual_page_numbers[number]}쪽: {len(text)}자")

PDF 52쪽 / 매뉴얼 47쪽: 782자
PDF 53쪽 / 매뉴얼 48쪽: 868자


### 2. 주제 경계 정하기

아래 여섯 제목은 이 두 페이지에서 직접 확인한 제목입니다. `collect_topics()`는 제목부터 다음 제목 전까지를 모으는 함수입니다. 페이지가 바뀌더라도 새 제목이 나오기 전까지는 앞 주제의 설명으로 이어 붙입니다.

페이지 아래 장 이름·쪽수·따로 떨어진 글머리표는 검색용 본문에서 제외합니다. 숫자는 그림의 조작 번호일 수도 있으므로 모든 숫자를 지우는 규칙은 사용하지 않습니다. 이 단계의 꼬리말 규칙은 확인한 두 페이지에만 적용합니다.

In [2]:
# [프로젝트 추가 기록] PDF 52~53쪽의 확인한 여섯 제목으로 주제 경계를 정하고 이어지는 문장을 묶습니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
topic_titles = [
    "엔진룸 점검",
    "규격 타이어 장착 및 타이어 공기압 수시 점검",
    "클러스터 및 페달류 점검",
    "올바른 운전 자세",
    "좌석, 스티어링 휠, 미러 조정",
    "운전석 주변 점검",
]

# 끝부분에 확인한 장 이름·쪽수와 글머리표만 있을 때 꼬리말로 처리합니다.
footer_patterns = {
    52: r"\n안전 및 주의 사항\n47\n(?:[•\s])*\Z",
    53: r"\n2\n48\n(?:[•\s])*\Z",
}
page_bodies = {}
for number, text in page_texts.items():
    footer = re.search(footer_patterns[number], text)
    if footer is None:
        raise ValueError(f"PDF {number}쪽 꼬리말 모양이 바뀌었습니다. 원문을 먼저 확인하세요.")
    page_bodies[number] = text[:footer.start()]

def collect_topics(page_bodies, titles):
    """확인한 제목으로 원문을 나누고, 다음 페이지의 이어지는 글도 같은 주제로 묶습니다."""
    title_pattern = re.compile(r"^(?:" + "|".join(re.escape(title) for title in titles) + r")$", re.MULTILINE)
    topics = []
    active_topic = None

    def append_segment(topic, number, text, start, end):
        """원문의 해당 구간과 위치를 기록합니다. start와 end는 글자 위치입니다."""
        if end > start:
            topic["segments"].append({
                "pdf_page_number": number,
                "start": start,
                "end": end,
                "text": text[start:end],
            })

    for number, text in page_bodies.items():
        matches = list(title_pattern.finditer(text))
        first_title_start = matches[0].start() if matches else len(text)

        # 다음 페이지 첫 제목 앞의 글은 이전 페이지에서 시작한 주제에 붙입니다.
        if first_title_start > 0:
            if active_topic is None:
                raise ValueError(f"PDF {number}쪽 첫 부분의 주제를 확인해야 합니다.")
            append_segment(active_topic, number, text, 0, first_title_start)

        for index, match in enumerate(matches):
            if active_topic is not None:
                topics.append(active_topic)
            active_topic = {"title": match.group(), "segments": []}
            end = matches[index + 1].start() if index + 1 < len(matches) else len(text)
            append_segment(active_topic, number, text, match.start(), end)

    if active_topic is not None:
        topics.append(active_topic)
    return topics

topic_groups = collect_topics(page_bodies, topic_titles)
for topic in topic_groups:
    pages = [segment["pdf_page_number"] for segment in topic["segments"]]
    print(topic["title"], "| PDF 쪽:", pages)

엔진룸 점검 | PDF 쪽: [52]
규격 타이어 장착 및 타이어 공기압 수시 점검 | PDF 쪽: [52]
클러스터 및 페달류 점검 | PDF 쪽: [52, 53]
올바른 운전 자세 | PDF 쪽: [53]
좌석, 스티어링 휠, 미러 조정 | PDF 쪽: [53]
운전석 주변 점검 | PDF 쪽: [53]


### 3. 검색용 글과 그림 연결하기

`raw_text`는 잘라 온 원문이고, `content`는 검색에 사용할 글입니다. 경고·주의·참고 표시는 읽을 수 있는 글자로 바꾸지만, 문장의 의미나 표현은 바꾸지 않습니다. 그림은 앞서 확인한 `I2.jpg`를 엔진룸 점검에만 붙입니다.

`source_spans`는 각 글 조각을 원문에서 다시 찾을 수 있게 하는 위치 정보입니다. 이미지의 `local_path`가 `None`이면 아직 이미지 파일로 저장하지 않았다는 뜻입니다.

In [3]:
# [프로젝트 추가 기록] PDF 52~53쪽의 확인한 여섯 제목으로 주제 경계를 정하고 이어지는 문장을 묶습니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
def prepare_search_text(raw_text):
    """원문 문장은 유지하고, 확인한 기호와 불필요한 이미지 코드만 정리합니다."""
    text = raw_text
    for old, new in {"҃Ҋ": "[경고]", "઱੄": "[주의]", "ӝ": "[참고]"}.items():
        text = text.replace(old, new)

    # 이미지 내부 코드는 한 줄 전체가 코드일 때만 제외합니다.
    text = re.sub(r"^[12]C_[A-Za-z0-9_]+\s*$", "", text, flags=re.MULTILINE)
    lines = [re.sub(r"[ \t]+", " ", line).strip() for line in text.splitlines()]
    return "\n".join(line for line in lines if line)

# 이름만 같은 다른 페이지의 그림을 연결하지 않도록 페이지와 이름을 함께 확인합니다.
engine_image = next((image for image in reader.pages[51].images if image.name == "I2.jpg"), None)
if engine_image is None:
    raise ValueError("PDF 52쪽의 I2.jpg 그림을 찾지 못했습니다.")
image_path = project_folder / "data" / "zzong_santafe_lag" / "images" / "pdf_page_52" / engine_image.name
engine_image_part = {
    "name": engine_image.name,
    "size": engine_image.image.size,
    "pdf_page_number": 52,
    "local_path": image_path.relative_to(project_folder).as_posix() if image_path.exists() else None,
    "description": "증기가 올라오는 용기 위로 손이 향해 있고 X 표시가 있다. 냉각수가 뜨거울 때 보조 탱크 캡을 열지 말라는 주의사항을 보여준다.",
}

topic_records = []
for topic in topic_groups:
    segments = topic["segments"]
    raw_text = "\n".join(segment["text"] for segment in segments)
    content = prepare_search_text(raw_text)
    image_parts = [dict(engine_image_part)] if topic["title"] == "엔진룸 점검" else []

    # 확인한 그림 설명은 그 그림에 해당하는 주제에만 붙입니다.
    if image_parts:
        content += "\n[그림 설명] " + image_parts[0]["description"]

    source_pages = list(dict.fromkeys(segment["pdf_page_number"] for segment in segments))
    metadata = {
        "title": topic["title"],
        "pdf_page_number": source_pages[0],
        "manual_page_number": manual_page_numbers[source_pages[0]],
        "source_pages": source_pages,
        "content_type": "figure_with_text" if image_parts else "instruction_text",
        "chunking_method": "confirmed_topic_titles",
        "source_spans": [
            {
                "pdf_page_number": segment["pdf_page_number"],
                "manual_page_number": manual_page_numbers[segment["pdf_page_number"]],
                "start": segment["start"],
                "end": segment["end"],
            } for segment in segments
        ],
    }
    if len(source_pages) > 1:
        metadata["continued_on_pdf_page"] = source_pages[1]
        metadata["continuation_manual_page_number"] = manual_page_numbers[source_pages[1]]

    topic_records.append({
        "content": content,
        "raw_text": raw_text,
        "metadata": metadata,
        "image_parts": image_parts,
    })

for record in topic_records:
    print(record["metadata"]["title"], "| PDF 쪽:", record["metadata"]["source_pages"],
          "| 검색용 글:", len(record["content"]), "자 | 이미지:", len(record["image_parts"]))

엔진룸 점검 | PDF 쪽: [52] | 검색용 글: 481 자 | 이미지: 1
규격 타이어 장착 및 타이어 공기압 수시 점검 | PDF 쪽: [52] | 검색용 글: 330 자 | 이미지: 0
클러스터 및 페달류 점검 | PDF 쪽: [52, 53] | 검색용 글: 162 자 | 이미지: 0
올바른 운전 자세 | PDF 쪽: [53] | 검색용 글: 203 자 | 이미지: 0
좌석, 스티어링 휠, 미러 조정 | PDF 쪽: [53] | 검색용 글: 319 자 | 이미지: 0
운전석 주변 점검 | PDF 쪽: [53] | 검색용 글: 177 자 | 이미지: 0


### 4. 원문 누락과 그림 연결 확인

확인할 것은 청크 수보다 내용입니다. 원문 구간이 빠지거나 중복되지 않았는지, 경고문이 남아 있는지, 두 페이지의 클러스터 설명이 이어지는지 확인합니다. 아래 검사는 이 실험 구간의 문제를 찾기 위한 것이며, 전체 779쪽의 청킹이 완성됐다는 뜻은 아닙니다.

In [4]:
# [프로젝트 추가 기록] PDF 52~53쪽의 확인한 여섯 제목으로 주제 경계를 정하고 이어지는 문장을 묶습니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
# 확인한 제목 여섯 개가 한 번씩 나오도록 합니다.
assert [record["metadata"]["title"] for record in topic_records] == topic_titles

# 각 페이지의 모든 본문 구간이 빠짐없이 한 번씩 사용됐는지 확인합니다.
for number, body in page_bodies.items():
    spans = sorted(
        [segment for topic in topic_groups for segment in topic["segments"]
         if segment["pdf_page_number"] == number],
        key=lambda segment: segment["start"],
    )
    assert "".join(segment["text"] for segment in spans) == body
    assert all(left["end"] == right["start"] for left, right in zip(spans, spans[1:]))
    for segment in spans:
        assert segment["text"] == page_texts[number][segment["start"]:segment["end"]]

by_title = {record["metadata"]["title"]: record for record in topic_records}
engine_record = by_title["엔진룸 점검"]
tire_record = by_title["규격 타이어 장착 및 타이어 공기압 수시 점검"]
cluster_record = by_title["클러스터 및 페달류 점검"]

# 서로 다른 주제를 섞거나 경고문을 잘라 버리지 않았는지 확인합니다.
assert "규격 타이어" not in engine_record["content"]
assert "[경고]" in engine_record["content"]
assert "화상을 입을 수 있습니다." in engine_record["content"]
assert "스탠딩 웨이브" in tire_record["content"]
assert cluster_record["metadata"]["source_pages"] == [52, 53]
assert "연료량이 충분한지" in cluster_record["content"]
assert "페달을 가볍게 밟아" in cluster_record["content"]
assert "[경고]" in by_title["좌석, 스티어링 휠, 미러 조정"]["content"]
assert len(engine_record["image_parts"]) == 1
assert all(not record["image_parts"] for record in topic_records if record is not engine_record)

print("주제 기록 수:", len(topic_records))
print("원문 본문 누락/중복:", "없음")
print("엔진룸 경고문:", "유지")
print("클러스터 점검 출처:", cluster_record["metadata"]["source_pages"])
print("그림 연결:", "엔진룸 점검에만 1개")
print("\n[두 페이지가 이어진 검색용 글]")
print(cluster_record["content"])

주제 기록 수: 6
원문 본문 누락/중복: 없음
엔진룸 경고문: 유지
클러스터 점검 출처: [52, 53]
그림 연결: 엔진룸 점검에만 1개

[두 페이지가 이어진 검색용 글]
클러스터 및 페달류 점검
클러스터 점검
클러스터에 표시된 각종 정보의 이상 유무를 확인하고 연료량이 충분한지 점검하십시오.
페달류 점검
페달을 가볍게 밟아 이상 유무를 확인하십시오. 평소 사용할 때보다 간극이 더 길거나 짧아지면
즉시 당사 직영 하이테크센터나 블루핸즈에서 점검을 받으십시오.


이 실험에서는 확인한 제목을 사용했기 때문에 주제와 그림을 정확하게 연결할 수 있습니다. 아직 전체 PDF의 제목을 자동으로 찾아 주는 규칙은 아닙니다. 긴 주제·표·안내도·여러 페이지에 걸친 절차는 별도 규칙이 필요합니다. 이후 이 기준을 넓힌 뒤 임베딩을 진행합니다.